# Experiments Section 9.1 — Table 2a and Figure 1a: AME

This notebook reproduces the AME part of Table 2 and Figure 1 in Section 9.1. It uses the nonlinear Gaussian DGP in the paper, two-fold cross-fitting, a common MLP outcome learner, and three representer estimators: Data-SMR, Time-SMR, and squared-loss Riesz regression.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
N_TRIALS = 200
N = 1000
N_FOLDS = 2
N_MC_TRUTH = 200000
HIDDEN_DIMS = (256, 256, 256)
OUTCOME_EPOCHS = 200
SCORE_STEPS = 4000
RATIO_STEPS = 4000
BATCH_SIZE = 256
DSM_SIGMA_MIN = 0.01
DSM_SIGMA_MAX = 1.0
SIGMA_EVAL = 0.01
AME_LOCAL_SHIFT = 0.05
INTEGRATION_STEPS = 128
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Table 2a: AME performance metrics"
FIGURE_TITLE = "Figure 1a: AME estimation errors"

In [ ]:

def mu_function(x):
    x = np.asarray(x)
    return (
        1.0 + x[:, 0]
        + 0.1 * x[:, 0] ** 2
        + 2.0 * np.sin(x[:, 0])
        + x[:, 1]
        + x[:, 0] * x[:, 1]
        + x[:, 2] ** 2
        + x[:, 2] ** 3
    )


def partial_d_mu(x):
    x = np.asarray(x)
    return 1.0 + 0.2 * x[:, 0] + 2.0 * np.cos(x[:, 0]) + x[:, 1]


def sample_x(n, seed):
    rng = np.random.default_rng(seed)
    sigma = np.array([[1.0, 0.1, 0.1], [0.1, 1.0, 0.1], [0.1, 0.1, 1.0]])
    return rng.multivariate_normal(np.zeros(3), sigma, size=n).astype("float32")


def sample_observations(n, seed):
    rng = np.random.default_rng(seed)
    x = sample_x(n, seed)
    y = mu_function(x) + rng.normal(size=n)
    return x.astype("float32"), y.astype("float32")


def shift_x(x, delta):
    out = np.asarray(x, dtype="float32").copy()
    out[:, 0] += float(delta)
    return out


def true_ame(n_mc, seed=999):
    x = sample_x(n_mc, seed)
    return float(np.mean(partial_d_mu(x)))


def true_ape(delta, n_mc, seed=777):
    x = sample_x(n_mc, seed)
    return float(np.mean(mu_function(shift_x(x, delta)) - mu_function(shift_x(x, -delta))))


def summarize_trials(df, group_cols):
    return (
        df.groupby(group_cols)
        .agg(
            trials=("estimate", "count"),
            truth=("truth", "mean"),
            bias=("error", "mean"),
            mse=("error", lambda s: float(np.mean(np.square(s)))),
            coverage=("covered", "mean"),
            avg_se=("se", "mean"),
        )
        .reset_index()
    )


In [ ]:

TRUE_AME = true_ame(N_MC_TRUTH)
METHODS = ["Data-SMR", "Time-SMR", "Riesz reg."]
print("True AME:", TRUE_AME)


def alpha_time_smr_ame(x_train, x_eval, seed):
    # Local-shift construction: (r_{+h}(x)-r_{-h}(x))/(2h) approximates -partial_d log p0(x).
    x_q_plus = shift_x(x_train, AME_LOCAL_SHIFT)
    x_q_minus = shift_x(x_train, -AME_LOCAL_SHIFT)
    m_plus = smr.fit_time_smr_dre_infinity(
        x_q_plus, x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS,
        batch_size=BATCH_SIZE, seed=seed, device=DEVICE
    )
    m_minus = smr.fit_time_smr_dre_infinity(
        x_q_minus, x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS,
        batch_size=BATCH_SIZE, seed=seed + 17, device=DEVICE
    )
    log_plus = smr.log_ratio_from_time_score(m_plus, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_train, device=DEVICE)
    log_minus = smr.log_ratio_from_time_score(m_minus, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_train, device=DEVICE)
    r_plus = np.exp(np.clip(log_plus.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    r_minus = np.exp(np.clip(log_minus.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    return (r_plus - r_minus) / (2.0 * AME_LOCAL_SHIFT)


def estimate_ame_trial(seed):
    x, y = sample_observations(N, seed)
    rows = []
    for method in METHODS:
        score_values = np.zeros(N)
        for train_idx, test_idx in smr.crossfit_splits(N, n_folds=N_FOLDS, seed=seed):
            x_train, y_train = x[train_idx], y[train_idx]
            x_test, y_test = x[test_idx], y[test_idx]
            outcome = smr.fit_outcome_net(
                x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS,
                batch_size=BATCH_SIZE, seed=seed, device=DEVICE
            )
            gamma_hat = smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
            m_gamma = smr.partial_d_outcome(outcome, x_test, coordinate=0, device=DEVICE).reshape(-1)
            if method == "Data-SMR":
                score_model = smr.fit_data_smr_score_dsm(
                    x_train, hidden_dims=HIDDEN_DIMS, n_steps=SCORE_STEPS,
                    batch_size=BATCH_SIZE, sigma_min=DSM_SIGMA_MIN, sigma_max=DSM_SIGMA_MAX,
                    seed=seed, device=DEVICE
                )
                alpha_hat = -smr.eval_data_score_d(score_model, x_test, sigma_eval=SIGMA_EVAL, device=DEVICE).reshape(-1)
            elif method == "Time-SMR":
                alpha_hat = alpha_time_smr_ame(x_train, x_test, seed)
            else:
                alpha_model = smr.fit_sq_riesz_ame(
                    x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS,
                    batch_size=BATCH_SIZE, seed=seed, device=DEVICE
                )
                alpha_hat = smr.eval_scalar_net(alpha_model, x_test, device=DEVICE).reshape(-1)
            score_values[test_idx] = m_gamma + alpha_hat * (y_test - gamma_hat)
        est = smr.wald_interval(score_values)
        rows.append({
            "method": method,
            "estimate": est.estimate,
            "se": est.se,
            "ci_low": est.ci_low,
            "ci_high": est.ci_high,
            "truth": TRUE_AME,
            "error": est.estimate - TRUE_AME,
            "covered": est.ci_low <= TRUE_AME <= est.ci_high,
        })
    return rows

trial_rows = []
for trial in range(N_TRIALS):
    trial_rows.extend([{**row, "trial": trial} for row in estimate_ame_trial(RANDOM_SEED + trial)])
ame_results = pd.DataFrame(trial_rows)
print(TABLE_TITLE)
display(summarize_trials(ame_results, ["method"]))


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
methods = list(ame_results["method"].unique())
ax.boxplot([ame_results.loc[ame_results["method"] == m, "error"] for m in methods], labels=methods, showfliers=False)
ax.axhline(0.0, linestyle="--")
ax.set_title(FIGURE_TITLE)
ax.set_ylabel("estimate minus truth")
fig.tight_layout()
plt.show()
